# Executing Quality Checks locally

This notebook runs the Reportnet 3 Quality Checks of the *River Basin Districts and Competent Authorities* dataflow against the deliverables produced by the prefill notebook, without submitting anything to Reportnet 3.

The QCs are stored in the `qc` schema of `wise_rbdsuca.duckdb` (built from the `.sql` files under `sql/qc/`), and they address the reported data through the same schema names Reportnet 3 uses:

| schema | resolves to |
| --- | --- |
| `descriptive_reporting` | the exported SQLite database |
| `spatial_reporting` | the exported GeoPackage, read through `ST_Read` so geometries are typed |
| `reference` | the reference datasets published in the data lake |

> `wise_rbdsuca.duckdb` ships precompiled - no build step needed. Run the prefill notebook first so `output/<CC>/` has the SQLite + GeoPackage this notebook attaches.

In [1]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Initial setup

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wise_local_qc.py").exists())
sys.path.insert(0, str(ROOT))

import ipywidgets as widgets

import wise_local_qc as wq

con = wq.connect(ROOT / "wise_rbdsuca.duckdb")
wq.current_parameters(con)

{'country_code': 'AT', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 2. Country selection

`Reference cycle` is the cycle the reference QCs compare the submission against. Choosing a cycle other than the reported one is a quick way to make the reference QCs produce findings.

In [2]:
countries_selection = widgets.Dropdown(options=wq.COUNTRIES, value="AT", description="Country:")
cycle_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Cycle:")
reference_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Reference:")
widgets.VBox([countries_selection, cycle_selection, reference_selection])

In [3]:
wq.set_parameters(
    con,
    country_code=countries_selection.value,
    cycle_year=cycle_selection.value,
    reference_cycle_year=reference_selection.value,
)

output_dir = ROOT / "output" / countries_selection.value
sqlite_path = output_dir / "RiverBasinDistrictsAndCompetentAuthorities.sqlite"
geopackage_path = output_dir / "RiverBasinDistrict.gpkg"

wq.attach_reporting(con, sqlite_path, geopackage_path)
wq.current_parameters(con)

{'country_code': 'ES', 'cycle_year': '2022', 'reference_cycle_year': '2016'}

## 3. Available tables

The reference views come from the catalog, the reporting views from the files just attached.

In [4]:
con.sql(
    """
    SELECT schema_name, view_name AS table_name
    FROM duckdb_views()
    WHERE schema_name IN ('reference', 'descriptive_reporting', 'spatial_reporting')
    ORDER BY schema_name, view_name
    """
).df()

,schema_name,table_name
0,descriptive_reporting,CompetentAuthority
1,descriptive_reporting,RiverBasinDistrictCompetentAuthority
2,reference,Country
3,reference,RiverBasinDistrictWFD
4,spatial_reporting,RiverBasinDistrict


In [17]:
con.sql("SELECT * EXCLUDE (geom) FROM spatial_reporting.RiverBasinDistrict").df()

,inspireIdLocalId,inspireIdNamespace,inspireIdVersionId,thematicIdIdentifier,thematicIdIdentifierScheme,beginLifespanVersion,endLifespanVersion,predecessorsIdentifier,predecessorsIdentifierScheme,successorsIdentifier,...,nameLanguage,designationPeriodBegin,designationPeriodEnd,zoneType,specialisedZoneType,legalBasisName,legalBasisLink,legalBasisLevel,link,record_id
0,ITA2018,RiverBasinDistrict_IT_20220101,1,ITA2018,euRBDCode,2018-10-15,None,ITA,euRBDCode,None,...,ita,2021-12-22,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,7b27042d-01fb-3960-d17d-57d358d31ace
1,ITB2018,RiverBasinDistrict_IT_20220101,1,ITB2018,euRBDCode,2018-10-15,None,"ITA,ITB,ITC",euRBDCode,None,...,ita,2021-12-21,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,816d7820-78bc-4a5b-7912-e75aa92bccd6
2,ITC2018,RiverBasinDistrict_IT_20220101,1,ITC2018,euRBDCode,2018-10-15,None,"ITC,ITD",euRBDCode,None,...,ita,2021-12-20,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,08b2d9b3-6ce9-c3a2-26ed-4e4cda6863d5
3,ITE2018,RiverBasinDistrict_IT_20220101,1,ITE2018,euRBDCode,2018-10-15,None,"ITC,ITE",euRBDCode,None,...,ita,2021-12-20,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,02397929-0961-bbf4-0827-5fe17df66397
4,ITF2018,RiverBasinDistrict_IT_20220101,1,ITF2018,euRBDCode,2018-10-15,None,ITF,euRBDCode,None,...,ita,2010-01-01,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,a793c9d6-c887-3ca1-e9a3-3a6eb90ff1b1
5,ITG2018,RiverBasinDistrict_IT_20220101,1,ITG2018,euRBDCode,2018-10-15,None,ITG,euRBDCode,None,...,ita,2010-01-01,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,91832766-7f6c-ac74-b0f3-34a464abdd9a
6,ITH2018,RiverBasinDistrict_IT_20220101,1,ITH2018,euRBDCode,2018-10-15,None,ITH,euRBDCode,None,...,ita,2010-01-01,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,b54a26df-80cf-7ad0-9ce5-236a3d1ddc6a


## 4. Available Quality Checks

In [18]:
wq.list_qcs(con)

,code,table_name,error_level,description
0,R003,SPATIAL_RiverBasinDistrict,ERROR,endLifespanVersion must be later than beginLif...
1,R011,SPATIAL_RiverBasinDistrict,ERROR,Creation objects must not report predecessors.
2,R014,SPATIAL_RiverBasinDistrict,ERROR,predecessorsIdentifierScheme must be reported ...
3,R015,SPATIAL_RiverBasinDistrict,ERROR,Each predecessor identifier must match an exis...
4,R016,SPATIAL_RiverBasinDistrict,ERROR,Deletion and noChange objects must not report ...
5,R022,SPATIAL_RiverBasinDistrict,ERROR,Change objects must report inspireIdVersionId.
6,R026,SPATIAL_RiverBasinDistrict,ERROR,changeCode and splitting objects must report e...
7,R027,SPATIAL_RiverBasinDistrict,ERROR,Predecessor identifiers must not be repeated w...
8,R028,SPATIAL_RiverBasinDistrict,ERROR,Objects with wiseEvolutionType creation or noC...
9,R029,SPATIAL_RiverBasinDistrict,ERROR,A successorsIdentifier must correspond to an e...


## 5. Execute the Quality Checks

A QC returns the offending records, so an empty result means the check passed.

In [5]:
summary, results = wq.run_qcs(con)
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,error_level,status,records,description,message
0,R003,ERROR,PASSED,0,endLifespanVersion must be later than beginLif...,
1,R011,ERROR,PASSED,0,Creation objects must not report predecessors.,
2,R014,ERROR,PASSED,0,predecessorsIdentifierScheme must be reported ...,
3,R015,ERROR,ERROR,0,Each predecessor identifier must match an exis...,BinderException: Binder Error: No function mat...
4,R016,ERROR,PASSED,0,Deletion and noChange objects must not report ...,
5,R022,ERROR,PASSED,0,Change objects must report inspireIdVersionId.,
6,R026,ERROR,PASSED,0,changeCode and splitting objects must report e...,
7,R027,ERROR,ERROR,0,Predecessor identifiers must not be repeated w...,InvalidInputException: Invalid Input Error: Un...
8,R028,ERROR,PASSED,0,Objects with wiseEvolutionType creation or noC...,
9,R029,ERROR,ERROR,0,A successorsIdentifier must correspond to an e...,InvalidInputException: Invalid Input Error: Un...


In [7]:
wq.write_qc_results(results, output_dir / "qc_results")

[]


## 6. AI-assisted interpretation of the failures

For every QC with status `FAILED` or `ERROR`, ask an LLM to explain what happened, its
likely root cause and recommended actions - built from the QC's own description, its
SQL and either a sample of the offending records (`FAILED`) or the exception message
(`ERROR`), so no separate documentation has to be maintained per QC.

Requires a provider to be configured through environment variables (see
`ai_explain.py`'s docstring): `AI_EXPLAIN_PROVIDER` defaults to `openai`, which only
needs `OPENAI_API_KEY` in a `.env` file (see `.env.example`).

In [21]:
import ai_explain
from IPython.display import Markdown, display

display(Markdown(ai_explain.explain_failures(con, summary, results)))

### R015 (ERROR - execution failed)

**What failed**: The DuckDB SQL query failed due to a type mismatch error when trying to use the `FLATTEN` function with an array of strings.
**Root cause**: The `FLATTEN` function is being used with an array of strings (`VARCHAR[]`), but the function expects an array of arrays (`T[][]`).
**Likely reasons**:
* The `predecessorsIdentifier` column is not of the expected type.
* The `FLATTEN` function is not imported or aliased correctly.
* The `STR_SPLIT_REGEX` function is not correctly splitting the string into an array of strings.
**Recommended actions**:
* Check the data type of the `predecessorsIdentifier` column in the `RiverBasinDistrict` table.
* Add explicit type casts to the `FLATTEN` function to match the expected type.
* Verify that the `STR_SPLIT_REGEX` function is correctly splitting the string into an array of strings.

---

### R027 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The STR_SPLIT_REGEX function in the SQL query.
**Likely reasons**:
* The 'ALL' option in the STR_SPLIT_REGEX function is not a valid option.
* The 'ALL' option is a typo and should be replaced with a valid option, such as 'ALL' being replaced with 'ALL' being replaced with an empty string or a valid option.
* The 'ALL' option is not supported by the DuckDB SQL dialect.
* The 'ALL' option is not compatible with the version of DuckDB being used.
**Recommended actions**:
* Replace the 'ALL' option with a valid option, such as an empty string or a valid option.
* Check the DuckDB SQL documentation for the correct syntax of the STR_SPLIT_REGEX function.
* Verify that the version of DuckDB being used supports the STR_SPLIT_REGEX function with the 'ALL' option.

---

### R029 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the `successorsIdentifier` column.
**Likely reasons**:
* The `successorsIdentifier` column is not a string type, or it contains a value that is not a string.
* The regular expression pattern used in `STR_SPLIT_REGEX` is incorrect or incomplete.
* The `successorsIdentifier` column contains a value that is too long to be split by the regular expression.
**Recommended actions**:
* Check the data type and content of the `successorsIdentifier` column.
* Verify the regular expression pattern used in `STR_SPLIT_REGEX` is correct and complete.
* Adjust the SQL query to handle the `successorsIdentifier` column correctly.

---

### R032 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option in the `STR_SPLIT_REGEX` function.
**Root cause**: The `STR_SPLIT_REGEX` function in the `predecessorsIdentifier` column.
**Likely reasons**:
* The `ALL` option is not a valid regular expression option for `STR_SPLIT_REGEX`.
* The `ALL` option was used in the original SQL, but it's not a valid option for this function.
* The `ALL` option might have been introduced in a newer version of the function, but the QC's SQL is outdated.
**Recommended actions**:
* Check the documentation for the `STR_SPLIT_REGEX` function to see if the `ALL` option is valid.
* If not, replace `ALL` with a valid option, such as `''`.
* If the `ALL` option is valid, check if there are any other issues with the regular expression pattern.

---

### R035 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the line `TRIM(FLATTEN(STR_SPLIT_REGEX(predecessorsIdentifier, ',', 'ALL'))) AS onePredecessorIdentifier,`.
**Likely reasons**:
* The `STR_SPLIT_REGEX` function is not supported in the current DuckDB version.
* The regular expression option 'A' is not a valid option for the `STR_SPLIT_REGEX` function.
* The `predecessorsIdentifier` column contains a value that is not a valid regular expression.
**Recommended actions**:
* Check the DuckDB documentation for supported regular expression options and functions.
* Replace the `STR_SPLIT_REGEX` function with a supported function, such as `STR_SPLIT`.
* Verify the data in the `predecessorsIdentifier` column to ensure it does not contain invalid regular expressions.

---

### R046 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the `predecessorsIdentifier` column.
**Likely reasons**:
* The `predecessorsIdentifier` column is not exported for this dataflow.
* The `predecessorsIdentifier` column has a different data type than expected.
* The regular expression pattern used in `STR_SPLIT_REGEX` is incorrect.
**Recommended actions**:
* Check if `predecessorsIdentifier` is exported for this dataflow in the `spatial_reporting` schema.
* Verify the data type of `predecessorsIdentifier` and adjust the SQL accordingly.
* Review the regular expression pattern used in `STR_SPLIT_REGEX` and adjust it if necessary.

---

### RF004_WFD (ERROR, 7 record(s))

**What failed**: The WFD reporters have reported River Basin Districts (RBDs) with a `wiseEvolutionType` other than 'noChange' or 'change' that exist in the reference data.

**Root cause**: The `wiseEvolutionType` field does not match one of the allowed values ('noChange', 'change') in the SQL query.

**Likely reasons**:
* The reporter has not updated the `wiseEvolutionType` field for some RBDs.
* The reporter has used an incorrect or outdated value for `wiseEvolutionType`.
* The reporter has reported RBDs that have undergone a different type of evolution (e.g. merging, splitting) that is not accounted for in the `wiseEvolutionType` field.

**Recommended actions**:
* Review the reporting guidelines and ensure that the `wiseEvolutionType` field is correctly filled in for all RBDs.
* Check the reference data to ensure that it accurately reflects the evolution of the RBDs.
* If necessary, update the `wiseEvolutionType` field for the affected RBDs to reflect the correct evolution type.

---

### RF008 (ERROR - execution failed)

**What failed**: The execution of the QC RF008 failed due to an invalid regular expression option in the STR_SPLIT_REGEX function.
**Root cause**: The `ALL` option in the `STR_SPLIT_REGEX` function.
**Likely reasons**:
* The `ALL` option is not a valid option for the `STR_SPLIT_REGEX` function in DuckDB.
* The `ALL` option was used in the original SQL, but it's not a standard option for this function.
* The SQL was copied from a different database system that supports this option.
**Recommended actions**:
* Replace the `ALL` option with the standard option for splitting by comma, which is `','`.
* Check the DuckDB documentation for the correct options for the `STR_SPLIT_REGEX` function.
* Verify that the SQL is compatible with the current database system.

---

### RF013_WFD (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' used in the `STR_SPLIT_REGEX` function.
**Root cause**: The `STR_SPLIT_REGEX` function in the line `TRIM(FLATTEN(STR_SPLIT_REGEX(predecessorsIdentifier, ',', 'ALL'))) AS onePredecessorIdentifier`.
**Likely reasons**:
* The `STR_SPLIT_REGEX` function is not supported in the current DuckDB version.
* The regular expression option 'A' is not a valid option for the `STR_SPLIT_REGEX` function.
* The `STR_SPLIT_REGEX` function is being used with an incorrect syntax.
**Recommended actions**:
* Replace the `STR_SPLIT_REGEX` function with a supported function, such as `STR_SPLIT`.
* Check the DuckDB documentation for the correct syntax and options for the `STR_SPLIT_REGEX` function.
* Consider using a different approach to split the string, such as using the `SPLIT` function.

---

### RF014_WFD (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the `STR_SPLIT_REGEX` function.
**Root cause**: The `STR_SPLIT_REGEX` function in the line `TRIM(FLATTEN(STR_SPLIT_REGEX(predecessorsIdentifier, ',', 'ALL'))) AS onePredecessorIdentifier`.
**Likely reasons**:
* The `STR_SPLIT_REGEX` function is not properly configured or is using an invalid regular expression option.
* The `predecessorsIdentifier` column contains a value that is not a valid regular expression.
* The `ALL` option is not a valid option for the `STR_SPLIT_REGEX` function.
**Recommended actions**:
* Check the documentation for the `STR_SPLIT_REGEX` function to ensure it is being used correctly.
* Verify that the `predecessorsIdentifier` column contains valid values.
* Replace the `ALL` option with a valid option, such as `''` or `''`.

---

### S005 (ERROR - execution failed)

**What failed**: DuckDB raised a CatalogException because the scalar function `st_srid` is not found.
**Root cause**: `ST_SRID(geometry_polygon)`.
**Likely reasons**:
* Table `RiverBasinDistrict` not exported for this dataflow.
* Column `geometry_polygon` not present in the exported table.
* Missing spatial extension in the database.
**Recommended actions**:
* Check if `RiverBasinDistrict` is exported for this dataflow in the meta.datasets.
* Adjust the SQL to use the correct spatial function (e.g. `ST_SetCRS`).
* Confirm that this QC applies to this dataflow and that the spatial extension is properly configured.

---

### S006 (ERROR - execution failed)

**What failed**: The QC failed to execute due to a binder error, indicating that the column "geometry_polygon" is not found in the FROM clause.
**Root cause**: The column "geometry_polygon" in the SQL.
**Likely reasons**:
* The table "RiverBasinDistrict" does not have a column named "geometry_polygon" in the active dataflow.
* The column "geometry_polygon" was renamed in the active dataflow.
* The table "RiverBasinDistrict" does not have a geometry column in the active dataflow.
**Recommended actions**:
* Check the meta.datasets to confirm that the column "geometry_polygon" exists in the table "RiverBasinDistrict" for the active dataflow.
* Adjust the SQL to use the correct column name, if it was renamed.
* Confirm that this QC applies to the active dataflow and that the table "RiverBasinDistrict" has a geometry column.

---

### S013 (ERROR - execution failed)

**What failed**: The QC failed due to a binder error, indicating that the column "geometry_polygon" is not found in the FROM clause.
**Root cause**: The column "geometry_polygon" in the SQL.
**Likely reasons**:
* Table not exported for this dataflow.
* Column "geometry_polygon" was renamed.
* Missing extension in the table or column.
**Recommended actions**:
* Check the meta.datasets to confirm if the table "RiverBasinDistrict" and column "geometry_polygon" are exported for this dataflow.
* Adjust the SQL to use the correct column name or alias.
* Confirm that this QC applies to this dataflow and that the column "geometry_polygon" is indeed used in the dataflow.

---

### S017 (ERROR - execution failed)

**What failed**: The QC failed to execute due to a missing column reference in the FROM clause.
**Root cause**: The column "geometry_polygon" is referenced in the SQL, but it is not found in the FROM clause.
**Likely reasons**:
* Table "RiverBasinDistrict" does not have a column named "geometry_polygon" (check meta.datasets).
* Column "geometry_polygon" is renamed or aliased in the SQL, but the correct alias is not used.
* The table "RiverBasinDistrict" is not exported for this dataflow.
**Recommended actions**:
* Check the meta.datasets to confirm the column name and alias.
* Adjust the SQL to use the correct column name or alias.
* Confirm that this QC applies to this dataflow and that the table "RiverBasinDistrict" is exported.

---

### T229 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' used in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the `predecessorsIdentifier` column of the `RiverBasinDistrict` table.
**Likely reasons**:
* The `predecessorsIdentifier` column contains a value that is not a valid regular expression.
* The regular expression option 'A' is not supported by DuckDB.
* The `predecessorsIdentifier` column is not exported for this dataflow.
**Recommended actions**:
* Check the data type and format of the `predecessorsIdentifier` column.
* Verify that the regular expression option 'A' is not used in the `STR_SPLIT_REGEX` function.
* Check the dataflow's meta.datasets to confirm that the `predecessorsIdentifier` column is exported.

---

### T230 (ERROR - execution failed)

**What failed**: A duplicate CTE name "vSRIDs" was encountered in the SQL query.
**Root cause**: The second CTE named "vSRIDs" in the SQL query.
**Likely reasons**:
* The SQL query was copied and pasted incorrectly, resulting in duplicate CTE names.
* The SQL query was modified and the CTE name was not updated accordingly.
* The SQL query was generated by a tool or script that did not properly handle CTE names.
**Recommended actions**:
* Rename the second CTE to a unique name, e.g. "vSRIDs_2".
* Remove the duplicate CTE and merge its contents into the previous CTE.
* Review the SQL query for any other potential errors or issues.

---

### V005 (ERROR - execution failed)

**What failed**: The error occurred while running the `regexp_matches` function due to an invalid Perl operator in the regular expression.
**Root cause**: The regular expression in the `regexp` column of the `Data` CTE.
**Likely reasons**:
* The `getvariable('country_code')` function is returning a value that is causing the regular expression to be malformed.
* The regular expression is not correctly escaped or formatted.
* The `regexp_matches` function is not supported in DuckDB.
**Recommended actions**:
* Check the value of `getvariable('country_code')` and ensure it is correctly formatted.
* Review and correct the regular expression in the `regexp` column.
* Consider using a different function or approach to achieve the desired result.

---

### V040 (ERROR - execution failed)

**What failed**: The execution of the QC V040 failed due to an invalid Perl operator in the regular expression.
**Root cause**: The regular expression in the `regexp` column of the `Data` CTE.
**Likely reasons**:
* The `getvariable('country_code')` function is not returning a valid country code.
* The regular expression pattern is incorrect or incomplete.
* The `predecessorsIdentifier` column is not in the expected format.
**Recommended actions**:
* Check the value of `getvariable('country_code')` and ensure it is a valid country code.
* Verify the regular expression pattern and adjust it if necessary.
* Confirm that the `predecessorsIdentifier` column is in the correct format for this dataflow.

## 7. Debug a single Quality Check

Print the SQL as stored in the catalog, then run it on its own. To iterate on a check, edit the corresponding file under `sql/qc/`, re-run `python build_catalog.py` and re-run the cells below.

In [10]:
qc_selection = widgets.Dropdown(options=wq.list_qcs(con)["code"].tolist(), description="QC:")
qc_selection

Dropdown(description='QC:', options=('R003', 'R011', 'R014', 'R015', 'R016', 'R022', 'R026', 'R027', 'R028', '…

In [11]:
print(wq.qc_sql(con, qc_selection.value))

WITH BothDates AS (
  SELECT
    record_id,
    CAST(beginLifespanVersion AS DATE) AS beginDate,
    CAST(endLifespanVersion AS DATE) AS endDate
  FROM spatial_reporting.RiverBasinDistrict
  /* Avoids values not reported */
  WHERE
    COALESCE(beginLifespanVersion, '') <> ''
    AND COALESCE(endLifespanVersion, '') <> ''
)
SELECT
  record_id,
  beginDate,
  endDate
FROM BothDates
WHERE
  NOT beginDate IS NULL AND NOT endDate IS NULL AND endDate <= beginDate


In [12]:
wq.run_qc(con, qc_selection.value)

,record_id,beginDate,endDate


## 8. Optional: see the checks fire

Prefilled data is consistent by construction, so every QC passes. The cell below writes a deliberately broken copy of the GeoPackage - one overlapping River Basin District and one shifted `designationPeriodBegin` - and points `spatial_reporting` at it, which makes S016 and RF012_WFD fail.

Re-run the *Country selection* cell to go back to the untouched deliverable.

In [13]:
wq.build_sandbox(con, geopackage_path, output_dir / "sandbox" / "RiverBasinDistrict.gpkg")
summary, results = wq.run_qcs(con)
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,error_level,status,records,description,message
0,R003,ERROR,PASSED,0,endLifespanVersion must be later than beginLif...,
1,R011,ERROR,PASSED,0,Creation objects must not report predecessors.,
2,R014,ERROR,PASSED,0,predecessorsIdentifierScheme must be reported ...,
3,R015,ERROR,ERROR,0,Each predecessor identifier must match an exis...,BinderException: Binder Error: No function mat...
4,R016,ERROR,PASSED,0,Deletion and noChange objects must not report ...,
5,R022,ERROR,PASSED,0,Change objects must report inspireIdVersionId.,
6,R026,ERROR,PASSED,0,changeCode and splitting objects must report e...,
7,R027,ERROR,ERROR,0,Predecessor identifiers must not be repeated w...,InvalidInputException: Invalid Input Error: Un...
8,R028,ERROR,PASSED,0,Objects with wiseEvolutionType creation or noC...,
9,R029,ERROR,ERROR,0,A successorsIdentifier must correspond to an e...,InvalidInputException: Invalid Input Error: Un...


In [14]:
results["S016"]

,record_id,thematicIdIdentifier,overlapsWith
0,14856b2c-1e0f-d34f-b90d-d6b82e593820,DE1000,"DE2000, XX9999"
1,480d4da6-c834-4687-414c-7ad4a1544811,DE4000,DE5000
2,5a2b2930-05eb-7366-97a5-812d0a6680c2,DE5000,"DE6000, DE9500, DE9610"
3,42f3d992-cd17-29b7-45d2-fe0e1395455a,DE2000,"DE3000, DE4000, DE5000, XX9999"


In [8]:
con.close()